# Бленд: соединение двух половин

Итоговое решение. В архив едут **оба** движка, и каждую пару считают оба:

    итог(категория) = w · ранг(половина 2) + (1 − w) · ранг(половина 1)

Ранги берутся ВНУТРИ категории. Ранг, а не вероятность: шкалы логитов у двух
моделей разные, и логит-среднее без калибровки заметно хуже (замерено −0.0079).

    половина 1   ecup + чекпойнт k48b1        mmBERT-base,        доска 0.5419
    половина 2   src  + bge_10m_len576_B_all  bge-reranker-v2-m3, доска 0.5382
    бленд                                                         доска 0.5595

Синергия +0.0176 к лучшей половине — она и есть смысл всей конструкции: модели
из разных семейств ошибаются в разных местах.

## Про веса

Два итоговых решения различаются **только** значением `w`:

| вариант | `w` | доска |
|---|---|---|
| равные | 0.5 во всех категориях | 0.5577 |
| вычисленные | по одному на категорию, из разбивок доски | 0.5595 |

Веса не подбирались: `blend.tuned_weights()` выводит их формулой
`w = clip(0.5 + (AP₂ − AP₁) / 0.22, 0.12, 0.88)` из официальных
покатегорийных разбивок двух одиночных сабмитов — тех самых тел, что едут
в архив. Разбивки лежат в дереве (`src/utils/lb_bge_b_all.json`,
`src/utils/lb_k48b1.json`, с id сабмитов внутри) и при каждой сборке
сверяются с замороженной копией `TUNED_W`.

Разница +0.0018. Все веса лежат внутри (0.12, 0.88): `w = 0` или `1` — это уже
не смесь, а маршрутизация, другой приём с другими рисками.

Равный вариант держится вторым решением как страховка: он не зависит ни от
одного вычисленного числа.

## 1. Импорты

In [ ]:
import os
import pathlib
import sys

# Тетради лежат в корне репозитория — там же, где src/.
ROOT = pathlib.Path.cwd()
os.chdir(ROOT)
os.environ["ECUP_ROOT"] = str(ROOT)
sys.path.insert(0, str(ROOT))
print("корень:", ROOT)

from src import config
from src.utils import blend, packaging

## 2. Половина 1 — архив с движком `ecup`

Из него берутся движок и чекпойнт (внутри архива — `artifacts/ce_f0`).
Собирается в `01_mmbert.ipynb`, раздел 7; сюда достаточно положить готовый zip —
но собранный этим же репозиторием: у архивов прежней раскладки движок лежит в
`ecup/`, а не в `src/ecup/`, и сборка их не примет.

In [ ]:
HALF1_ZIP = config.SUBMISSIONS / 'submission_k48b1.zip'
assert HALF1_ZIP.exists(), f'нет {HALF1_ZIP} — соберите его в 01_mmbert.ipynb'
print(f'половина 1: {HALF1_ZIP.name}, {HALF1_ZIP.stat().st_size / 2**20:.0f} МБ')

## 3. Половина 2 — чекпойнт стадии B

In [ ]:
APPROACH = 'bge_10m_len576_B_all'
assert (config.ARTIFACTS / APPROACH).is_dir(), \
    f'нет весов {APPROACH} — обучите в 02_bge.ipynb'
print(f'половина 2: {APPROACH}')

## 4. Веса

Равные — по одному на все двадцать категорий.

In [ ]:
# Два варианта итогового решения. Меняется ровно одна строка ниже.
#
#   'равные'       w = 0.5 везде                                        0.5577
#   'вычисленные'  w из официальных разбивок двух одиночных сабмитов    0.5595
#
VARIANT = 'равные'

if VARIANT == 'равные':
    WEIGHTS = blend.equal_weights()
    NAME = 'blend_equal'
elif VARIANT == 'вычисленные':
    # w(c) = clip(0.5 + (AP2 - AP1) / 0.22, 0.12, 0.88) из src/utils/lb_*.json.
    WEIGHTS = blend.tuned_weights()
    NAME = 'blend_tuned'
else:
    raise SystemExit(f'неизвестный вариант {VARIANT!r}')

assert set(WEIGHTS) == set(blend.CATEGORIES), 'веса не на все 20 категорий'
print(f'{VARIANT}: весов {len(WEIGHTS)}, '
      f'от {min(WEIGHTS.values()):.3f} до {max(WEIGHTS.values()):.3f}')

## 5. Сборка и стенд

Запустите тетрадь дважды, меняя `VARIANT` — получатся оба итоговых архива.

Проверки две, и они про разное:

* `packaging.verify` запускает архив **в докере**, в официальном образе, на
  тысяче пар: не забыт ли файл, всё ли импортируется, не вырождается ли выход
  в константу. Времени она не меряет;
* `50_selftest.py` гоняет архив **на трёх этапах с настоящими лимитами**
  (1000 / 60 с, 115 тыс. / 360 с, 275 тыс. / 780 с), с каталогом решения только
  на чтение и заглушенной сетью, и печатает суммарное время. Запас меньше 15%
  он считает провалом.

Второй пропускать не стоит: у бленда два тела, и время — самое узкое место
конструкции. Докера на арендованной машине обычно нет, и тогда `verify` честно
сообщает об этом и пропускается — стенд с лимитами докера не требует.

In [ ]:
zip_blend = blend.build(WEIGHTS, approach=APPROACH,
                        name=NAME, half1_zip=HALF1_ZIP)
packaging.verify(zip_blend)
print('готово:', zip_blend)

In [ ]:
import subprocess, sys

# Стенд с лимитами: три этапа, каталог решения только на чтение, сеть заглушена.
subprocess.run([sys.executable, '-u', 'src/scripts/50_selftest.py',
                '--zip', str(zip_blend)], check=True)